# Fault Deformation & Seismic Hazard Toolkit
### Integration notebook — 2023 Kahramanmaraş earthquake sequence

This notebook walks through three independent GeoAI components built against the same real earthquake sequence, and ties them together around one question: **could combining cheap, wide-area geophysical signals reduce reliance on sparse in-situ instrumentation for hazard monitoring?**

This is the same framing thread as the geophysical inverse-theory / CPT project and the SLM (Supraglacial Lake Monitor) project in this portfolio — using satellite and public data to substitute for or augment sparse point-instrumentation, at a fraction of the cost.

| Stage | Signal | Technique | Headline result |
|---|---|---|---|
| 1 | Satellite radar (InSAR) | LiCSBAS SBAS time series | Clean coseismic velocity dipole across the fault |
| 2 | Public earthquake catalog | XGBoost classifier | PR-AUC 0.457 (~16x random baseline) |
| 3 | Public seismic waveforms | 1D conv autoencoder | 12.08 dB SNR improvement |

Each stage uses a genuinely different observation type — space-based deformation, catalog statistics, and ground-based waveforms — which is the point: no single signal alone tells the full hazard story, but together they triangulate it.

In [ ]:
from IPython.display import Image, display
import pandas as pd
from pathlib import Path

FIG_DIR = Path("results/figures")
DATA_DIR = Path("data")

---
## Stage 1: InSAR Fault Deformation

**Frame:** `116A_05207_252525` (COMET-LiCS, ascending track), covering the Adıyaman/Malatya segment of the East Anatolian Fault.

**Method:** Standard LiCSBAS pipeline (download → multilook → mask → unwrapping QC → loop closure QC → SBAS inversion → filtering). Two full days of QC investigation revealed and worked around a systematic issue with the 4 Feb epoch (failed loop closure across every pair using that date) and expected near-fault decorrelation immediately after the mainshock — both documented as findings, not hidden as failures.

**Result:** a clear positive/negative line-of-sight velocity dipole straddling the fault trace, with the two crustal blocks diverging by roughly 80–100mm across the earthquake window — the expected signature of left-lateral strike-slip rupture.

In [ ]:
display(Image(filename=str(FIG_DIR / "velocity_map.png")))

In [ ]:
display(Image(filename=str(FIG_DIR / "time_series_comparison.png")))

---
## Stage 2: Aftershock Density Forecaster

**Data:** USGS FDSNWS catalog, 30 days post-mainshock, 456 events (M≥3.4 practical completeness).

**Method:** Binary classification (following [DeVries et al. 2018, Nature](https://www.nature.com/articles/s41586-018-0438-y)) — does a 0.1° grid cell see ≥1 aftershock in a given time window? XGBoost classifier on distance-from-mainshock, grid location, and time-since-mainshock, built against the full grid (including true negatives) rather than only positive examples.

**Result:** ROC-AUC 0.933, PR-AUC 0.457 (~16x the random baseline given the 2.8% positive class rate). The model correctly identifies the fault-aligned corridor as high-risk, including spatially separate clusters away from the main rupture.

In [ ]:
display(Image(filename=str(FIG_DIR / "aftershock_forecast_comparison.png")))

---
## Stage 3: Seismic Waveform Denoiser

**Data:** Real waveforms from 3 stations (GE.EIL, IU.ANTO, IU.GNI) near the epicenter, pulled via ObsPy/EarthScope.

**Method:** Small 1D convolutional autoencoder. Genuinely paired noisy/clean recordings don't exist for this use case, so training pairs are synthetic: real waveforms cut into overlapping windows, with randomized-strength Gaussian noise added as input and the real window as the reconstruction target — standard practice when paired data isn't available.

**Result:** 12.08 dB mean SNR improvement on held-out test windows (-6.61 dB noisy input → +5.47 dB denoised output).

In [ ]:
display(Image(filename=str(FIG_DIR / "denoising_before_after.png")))

---
## Synthesis: why combine these three?

Each stage independently substitutes a cheap, wide-area, publicly available signal for something that traditionally requires dense in-situ instrumentation:

- **InSAR replaces GNSS networks** for measuring surface deformation over a wide area — a single satellite pass covers hundreds of km, versus discrete GNSS stations that only sample specific points.
- **The aftershock forecaster replaces expert manual hazard-zoning** — a data-driven model, retrained cheaply per-event, flags likely aftershock corridors directly from catalog data that's already public.
- **The denoiser extends the useful range of existing seismic stations** — a station whose raw signal is too noisy to interpret becomes usable, effectively increasing the density of the existing network without adding hardware.

None of these alone is a complete hazard-monitoring system, and none is presented here as one. But together, they sketch a workflow where **wide-area, cheap, publicly available signals do real diagnostic work that would otherwise require sparse, expensive point instrumentation** — the same substitution logic behind the CPT/geotechnical inversion project (ML regression replacing dense subsurface sampling) and SLM (CNN segmentation replacing manual/point-based lake surveying) elsewhere in this portfolio.

## Honest limitations, taken together

- Single-event proof of concept throughout — none of the three components has been validated on a second earthquake. The aftershock model in particular partly learned this specific fault's geometry, not a fully general distance-decay relationship.
- The InSAR analysis used one ascending track only (no cross-validation against descending), and no atmospheric correction (GACOS) was applied.
- The denoiser trained on only 5 real waveform traces from 3 stations — genuinely paired noisy/clean data doesn't exist, so training pairs are synthetic.
- This demonstrates workflow and technique fluency across three distinct data modalities, not a deployable, validated hazard-monitoring system.